# Inferenza Layer Normativi

## Principio metodologico

I livelli della piramide normativa **emergono dai dati**.
L'algoritmo identifica cluster di atti che condividono la stessa **funzione normativa**,
indipendentemente dall'argomento trattato.

Punto critico: due regolamenti che trattano lo stesso argomento ma hanno funzioni diverse
(uno quadro, uno tecnico-operativo) devono finire in cluster diversi. Per questo non usiamo
embedding del testo completo (che cattura il dominio tematico), ma **feature funzionali**
che catturano il ruolo normativo.

## Feature utilizzate

| Gruppo | Cosa cattura | Come si estrae |
|---|---|---|
| **Funzionali** | Ruolo normativo: chi adotta, con quale procedura, che obbligo crea | Regex + LLM (domande chiuse); `primary_obligation` separata per fonte (regex ×2, LLM ×0.5) |
| **Strutturali** | Posizione nella gerarchia della rete di citazioni (indegree, outdegree, citation_ratio, pagerank) | NetworkX sul grafo |
| **Semantiche ridotte** | Sfumature linguistiche residue non catturate da regex | Embedding su titolo + 200c preambolo |

## Pipeline

```
1. Feature strutturali dal grafo
2. Feature funzionali (regex → LLM per i campi mancanti)
3. Embedding su testo breve
4. Normalizzazione e concatenazione
5. UMAP: riduzione dimensionale
6. HDBSCAN: clustering — il numero di layer emerge dai dati
7. Ordinamento gerarchico dei cluster
8. Purezza, entropia, level_span per ogni atto
9. Validazione post-hoc
```

## Input / Output
- **Input**: `nodes_focal_texts.csv`, `edges_focal.csv`
- **Output**: `nodes_focal_layers.csv`

## 0. Setup e Parametri

In [48]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import numpy as np
import re, os, sys, json, time

sys.path.append('..')
from config_golden_power import MATERIA_NAME

# ── Percorsi ─────────────────────────────────────────────────────────────────
output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'nodes_focal_texts.csv')
edges_file      = os.path.join(output_path, 'edges_focal.csv')
output_file     = os.path.join(output_path, 'nodes_focal_layers.csv')
checkpoint_file = os.path.join(output_path, 'functional_features_checkpoint.csv')

# ── Parametri LLM ────────────────────────────────────────────────────────────
LLM_MODEL        = 'deepseek-chat'
LLM_DELAY        = 0.3
LLM_MAX_RETRIES  = 3
CHECKPOINT_EVERY = 50

# ── Parametri UMAP ───────────────────────────────────────────────────────────
UMAP_N_COMPONENTS = 15
UMAP_N_NEIGHBORS  = 20
UMAP_MIN_DIST     = 0.05
UMAP_RANDOM_STATE = 42

# ── Parametri HDBSCAN ────────────────────────────────────────────────────────
# min_cluster_size: dimensione minima per essere considerato un layer.
# 400 → 4-6 layer macro, più leggibili e allineati con Lamfalussy.
# Diminuire → più layer, più fini. Aumentare → meno layer, più generali.
HDBSCAN_MIN_CLUSTER_SIZE = 400
HDBSCAN_MIN_SAMPLES      = 50

# ── Soglie interpretazione ───────────────────────────────────────────────────
PURITY_THRESHOLD     = 0.70   # sotto questa soglia: atto ibrido
MEMBERSHIP_THRESHOLD = 0.15   # soglia per contare un layer nel level_span
SPAN_THRESHOLD       = 2      # level_span >= 2: potenzialmente patologico

# ── Pesi ordinamento gerarchico ───────────────────────────────────────────────
# Determinano quale segnale pesa di più nel decidere quale cluster è L1.
# I due pesi devono sommare a 1.0.
# RATIO:    citation_ratio medio del cluster (atti apicali citati-da-molti / cita-pochi)
# RECEIVED: citazioni inter-cluster ricevute (apicale = tutti lo citano)
# Cambiare questi valori e ri-eseguire la cella 9 non richiede di rieseguire il clustering.
HIERARCHY_W_RATIO    = 0.6
HIERARCHY_W_RECEIVED = 0.4
assert abs(HIERARCHY_W_RATIO + HIERARCHY_W_RECEIVED - 1.0) < 1e-9, "I pesi devono sommare a 1.0"

# ── Modello embedding ─────────────────────────────────────────────────────────
# all-mpnet-base-v2: 768d → PCA 50d. Più potente di MiniLM ma più lento (~2 min).
# Nota: anche con testo breve (200c) mpnet può catturare sfumature tematiche;
# questo è accettabile perché il suo peso nella feature matrix è bilanciato
# dalle feature funzionali e strutturali (×2).
EMBEDDING_MODEL = 'all-mpnet-base-v2'

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print(f"Hierarchy weights — ratio: {HIERARCHY_W_RATIO}  received: {HIERARCHY_W_RECEIVED}")

Input:  ..\data\output\golden_power\nodes_focal_texts.csv
Output: ..\data\output\golden_power\nodes_focal_layers.csv


## 1. Caricamento Dati e Feature Strutturali

Il grafo ha **29 tipi di relazione** con semantica normativa precisa.
Vengono sfruttati costruendo:
- **Grafo pesato** per tipo (BASED_ON=3, IMPLEMENTS=2.5, CITES=0.3, ecc.) → PageRank e HITS gerarchici
- **Contatori per tipo** (`based_on_received`, `amends_sent`, ecc.) → feature dirette
- **Feature aggregate** per famiglia (`hierarchical_authority`, `lateral_activity`)
- **Normalizzazione temporale**: ogni feature viene divisa per la media degli atti dello stesso anno,
  eliminando il bias per cui atti vecchi appaiono più centrali solo per anzianità
- **HITS**: decompone ogni nodo in authority (apicale) e hub (subordinato)
- **K-core**: posizione topologica nel nucleo della rete, indipendente dal tempo

In [11]:
import networkx as nx

nodes = pd.read_csv(input_file)
edges = pd.read_csv(edges_file)

print(f"Nodi:  {len(nodes)}")
print(f"Archi: {len(edges)}")

# Adatta nomi colonne archi
src_col  = 'Source' if 'Source' in edges.columns else edges.columns[0]
tgt_col  = 'Target' if 'Target' in edges.columns else edges.columns[1]
type_col = 'Type'   if 'Type'   in edges.columns else (':TYPE' if ':TYPE' in edges.columns else None)

if type_col:
    print(f"\nTipi di relazione presenti ({edges[type_col].nunique()}):")
    print(edges[type_col].value_counts().to_string())

# ── Famiglie di tipo per valore gerarchico ─────────────────────────────────────
# HIERARCHICAL_DOWN: A cita B come base/fondamento → B è sopra A
HIERARCHICAL_DOWN = {'BASED_ON', 'IMPLEMENTS', 'ADOPTS', 'PARTIALLY_ADOPTS'}
# LATERAL: stessa fascia gerarchica (modifica, corregge, completa)
LATERAL           = {'AMENDS', 'CORRECTS', 'COMPLETES', 'ADDS_TO',
                     'DOES_INSERTION', 'DOES_DELETION', 'DOES_REPLACEMENT',
                     'EXTENDS_APPLICATION', 'EXTENDS_VALIDITY', 'REPLACES',
                     'DOES_REPEAL', 'REESTABLISHES'}
# ABROGATION: A abroga B → stesso livello, A più recente
ABROGATION        = {'REPEALS', 'IMPLICITLY_REPEALS'}
# SOFT: segnale debole, direzione gerarchica incerta
SOFT              = {'CITES', 'RELATED_TO', 'RELATED_QUESTION_TO',
                     'INFLUENCES', 'DEROGATES', 'SUSPENDS', 'PARTIALLY_SUSPENDS',
                     'PROPOSES_TO_AMEND', 'DEFERS_APPLICATION', 'INCORPORATES'}
# INTERPRETIVE: solo sentenze CGUE
INTERPRETIVE      = {'INTERPRETES_AUTHORITATIVELY'}

# ── Pesi per grafo pesato ──────────────────────────────────────────────────────
# Peso proporzionale alla certezza gerarchica del segnale
EDGE_WEIGHTS = {
    'BASED_ON':                   3.0,
    'IMPLEMENTS':                 2.5,
    'ADOPTS':                     2.0,
    'PARTIALLY_ADOPTS':           2.0,
    'AMENDS':                     1.0,
    'CORRECTS':                   0.8,
    'REPEALS':                    1.0,
    'IMPLICITLY_REPEALS':         0.8,
    'COMPLETES':                  0.8,
    'DOES_REPLACEMENT':           0.8,
    'DOES_INSERTION':             0.5,
    'DOES_DELETION':              0.5,
    'DOES_REPEAL':                0.5,
    'EXTENDS_APPLICATION':        0.5,
    'EXTENDS_VALIDITY':           0.5,
    'REPLACES':                   0.8,
    'DEROGATES':                  0.5,
    'INTERPRETES_AUTHORITATIVELY':1.5,
    'CITES':                      0.3,
}
DEFAULT_WEIGHT = 0.3

# ── Costruzione grafi ─────────────────────────────────────────────────────────
# G_unweighted: per k-core e feature di conteggio
# G_weighted:   per PageRank e HITS gerarchici
G = nx.DiGraph()
G.add_nodes_from(nodes['Id'].tolist())

G_w = nx.DiGraph()
G_w.add_nodes_from(nodes['Id'].tolist())

for _, row in edges.iterrows():
    s, t = row[src_col], row[tgt_col]
    et   = row[type_col] if type_col else 'CITES'
    w    = EDGE_WEIGHTS.get(et, DEFAULT_WEIGHT)
    G.add_edge(s, t, edge_type=et)
    G_w.add_edge(s, t, weight=w)

# ── Feature di conteggio per tipo ─────────────────────────────────────────────
print("\nCalcolo feature strutturali tipate...")

# Contatori per nodo
type_counts = {nid: {} for nid in nodes['Id']}

for _, row in edges.iterrows():
    s, t = row[src_col], row[tgt_col]
    et   = row[type_col] if type_col else 'CITES'
    # ricevuto (in): s viene 'raggiunto' da t? No: in un arco s→t, s EMETTE, t RICEVE
    if t in type_counts:
        key_in = f"{et.lower()}_received"
        type_counts[t][key_in] = type_counts[t].get(key_in, 0) + 1
    if s in type_counts:
        key_out = f"{et.lower()}_sent"
        type_counts[s][key_out] = type_counts[s].get(key_out, 0) + 1

counts_df = pd.DataFrame.from_dict(type_counts, orient='index').fillna(0)
counts_df.index.name = 'Id'
counts_df = counts_df.reset_index()

nodes = nodes.merge(counts_df, on='Id', how='left')
# Riempi eventuali NaN (nodi senza archi)
count_cols = [c for c in nodes.columns if c.endswith('_received') or c.endswith('_sent')]
nodes[count_cols] = nodes[count_cols].fillna(0)

# ── Feature aggregate per famiglia ─────────────────────────────────────────────
# Autorità gerarchica: quanti atti si fondano/implementano su questo
nodes['hierarchical_authority'] = sum(
    nodes.get(f"{et.lower()}_received", pd.Series(0, index=nodes.index))
    for et in HIERARCHICAL_DOWN
)
# Subordinazione gerarchica: su quanti atti si fonda/implementa
nodes['hierarchical_subordination'] = sum(
    nodes.get(f"{et.lower()}_sent", pd.Series(0, index=nodes.index))
    for et in HIERARCHICAL_DOWN
)
# Attività laterale: modifiche emesse/ricevute
nodes['lateral_activity'] = sum(
    nodes.get(f"{et.lower()}_received", pd.Series(0, index=nodes.index)) +
    nodes.get(f"{et.lower()}_sent",     pd.Series(0, index=nodes.index))
    for et in LATERAL
)
# Ratio gerarchico: atti che si fondano su di me / atti su cui mi fondo
nodes['hierarchical_ratio'] = (
    nodes['hierarchical_authority'] /
    (nodes['hierarchical_subordination'] + 1)
)

# ── Indegree / outdegree generici (backward compat) ────────────────────────────
indegree  = dict(G.in_degree())
outdegree = dict(G.out_degree())
nodes['indegree']       = nodes['Id'].map(indegree).fillna(0)
nodes['outdegree']      = nodes['Id'].map(outdegree).fillna(0)
nodes['citation_ratio'] = nodes['indegree'] / (nodes['outdegree'] + 1)

# ── PageRank su grafo pesato ───────────────────────────────────────────────────
pagerank_w = nx.pagerank(G_w, alpha=0.85, max_iter=300, weight='weight')
nodes['pagerank'] = nodes['Id'].map(pagerank_w).fillna(0)

# ── HITS su grafo pesato ───────────────────────────────────────────────────────
# hub_score:       quanto questo atto punta ad authority importanti (subordinazione)
# authority_score: quanto questo atto è puntato da hub importanti (apicalità)
try:
    hits_hub, hits_auth = nx.hits(G_w, max_iter=500, normalized=True)
    nodes['hits_authority'] = nodes['Id'].map(hits_auth).fillna(0)
    nodes['hits_hub']       = nodes['Id'].map(hits_hub).fillna(0)
    # Ratio HITS: alta authority + basso hub → apicale
    nodes['hits_ratio'] = nodes['hits_authority'] / (nodes['hits_hub'] + 1e-9)
    print("HITS: ok")
except nx.PowerIterationFailedConvergence:
    print("HITS: non converge — impostati a 0")
    nodes['hits_authority'] = 0.0
    nodes['hits_hub']       = 0.0
    nodes['hits_ratio']     = 0.0

# ── K-core su grafo non diretto (posizione topologica) ────────────────────────
G_und = G.to_undirected()
G_und.remove_edges_from(nx.selfloop_edges(G_und))  # k-core non supporta self-loop
core_number = nx.core_number(G_und)
nodes['kcore'] = nodes['Id'].map(core_number).fillna(0)

# ── Normalizzazione temporale delle feature strutturali ───────────────────────
# Problema: indegree/pagerank dipendono dall'anzianità dell'atto.
# Un regolamento del 1990 ha indegree alto perché ha avuto 30 anni per
# essere citato — non necessariamente perché è più apicale di uno del 2020.
# Soluzione: dividere per la media degli atti dello stesso anno.
# Il risultato misura 'quanto è citato rispetto ai suoi contemporanei'.

if 'Year' in nodes.columns:
    year_col = 'Year'
elif 'year' in nodes.columns:
    year_col = 'year'
else:
    year_col = None

if year_col:
    for feat in ['indegree', 'pagerank', 'hits_authority', 'hierarchical_authority']:
        year_mean = nodes.groupby(year_col)[feat].transform('mean').replace(0, np.nan)
        nodes[f"{feat}_norm"] = (nodes[feat] / year_mean).fillna(1.0)
    print("Normalizzazione temporale: ok")
else:
    print("[SKIP] Colonna anno non trovata — normalizzazione temporale disabilitata")
    for feat in ['indegree', 'pagerank', 'hits_authority', 'hierarchical_authority']:
        nodes[f"{feat}_norm"] = nodes[feat]

# ── Riepilogo ─────────────────────────────────────────────────────────────────
print()
print("Feature strutturali calcolate:")
struct_report = [
    ('indegree',                  'archi totali ricevuti'),
    ('outdegree',                 'archi totali emessi'),
    ('citation_ratio',            'indegree / (outdegree+1)'),
    ('pagerank',                  'PageRank su grafo pesato per tipo'),
    ('hits_authority',            'HITS authority (apicalità)'),
    ('hits_hub',                  'HITS hub (subordinazione)'),
    ('hits_ratio',                'authority / hub'),
    ('kcore',                     'k-core (posizione topologica)'),
    ('hierarchical_authority',    'BASED_ON/IMPLEMENTS ricevuti'),
    ('hierarchical_subordination','BASED_ON/IMPLEMENTS emessi'),
    ('hierarchical_ratio',        'authority ger. / subordinazione ger.'),
    ('lateral_activity',          'modifiche/correzioni totali'),
    ('indegree_norm',             'indegree normalizzato per anno'),
    ('pagerank_norm',             'pagerank normalizzato per anno'),
    ('hits_authority_norm',       'hits_authority normalizzato per anno'),
    ('hierarchical_authority_norm','hierarchical_authority norm. per anno'),
]
for col, desc in struct_report:
    if col in nodes.columns:
        print(f"  {col:<35} mean={nodes[col].mean():.4f}  max={nodes[col].max():.4f}")

# Feature di conteggio per tipo più frequenti
print()
print("Top feature di conteggio per tipo (mean > 0.01):")
for c in sorted(count_cols):
    m = nodes[c].mean()
    if m > 0.01:
        print(f"  {c:<40} mean={m:.3f}  max={nodes[c].max():.0f}")

Nodi:  4904
Archi: 25743

Calcolo feature strutturali...
indegree  — min: 0  max: 298  mean: 5.1
outdegree — min: 0  max: 73  mean: 5.1


## 2. Feature Funzionali via Regex

Estrae autore, procedura, base TFUE e tipo di obbligo dai pattern testuali
inequivocabili. Il LLM (Step 3) riempirà solo i campi rimasti vuoti.

In [26]:
def extract_regex_features(title, preamble):
    title_s    = str(title)    if pd.notna(title)    else ''
    preamble_s = str(preamble) if pd.notna(preamble) else ''
    # Usa titolo + primi 600 caratteri del preambolo:
    # procedura e base giuridica sono sempre nelle prime righe
    text   = f"{title_s} {preamble_s[:600]}"
    text_u = text.upper()

    f = {'author': None, 'procedure': None,
         'tfeu_article': None, 'primary_obligation': None,
         'has_technical_annex': False}

    # ── Autore istituzionale (dal più specifico al più generale) ──────────────
    if re.search(r'JUDGMENT OF THE COURT|ORDER OF THE COURT|OPINION OF THE COURT', text_u):
        f['author'] = 'Court'
    elif re.search(r'COMMISSION DELEGATED REGULATION|DELEGATED REGULATION\s*\(EU\)', text_u):
        f['author'] = 'Commission_delegated'
    elif re.search(r'COMMISSION IMPLEMENTING REGULATION|IMPLEMENTING REGULATION\s*\(EU\)', text_u):
        f['author'] = 'Commission_implementing'
    elif re.search(r'EUROPEAN PARLIAMENT AND.{0,30}COUNCIL|COUNCIL AND.{0,30}EUROPEAN PARLIAMENT', text_u):
        f['author'] = 'EP_Council'
    elif re.search(r'COUNCIL OF THE EUROPEAN UNION|THE COUNCIL,|THE COUNCIL ACTING', text_u):
        f['author'] = 'Council'
    elif re.search(r'COMMISSION RECOMMENDATION|COMMISSION COMMUNICATION|COMMISSION DECISION', text_u):
        f['author'] = 'Commission_other'
    elif re.search(r'TREATY|TFEU|TREATY ON THE FUNCTIONING', text_u):
        f['author'] = 'Treaty'

    # ── Procedura adottiva ────────────────────────────────────────────────────
    if re.search(r'ORDINARY LEGISLATIVE PROCEDURE|CO-DECISION PROCEDURE', text_u):
        f['procedure'] = 'ordinary_legislative'
    elif re.search(r'PURSUANT TO ARTICLE 29[01]|DELEGATED BY|EMPOWERED BY ARTICLE', text_u):
        f['procedure'] = 'delegated_implementing'
    elif re.search(r'ARTICLE 258|ARTICLE 260|INFRINGEMENT PROCEEDINGS|HAS FAILED TO FULFIL', text_u):
        f['procedure'] = 'infringement'
    elif re.search(r'HEREBY RECOMMENDS|NON-BINDING|SHOULD BE UNDERSTOOD', text_u):
        f['procedure'] = 'recommendation'
    elif f['author'] == 'Court':
        f['procedure'] = 'judgment'
    elif f['author'] == 'Treaty':
        f['procedure'] = 'treaty'

    # ── Base giuridica TFUE ───────────────────────────────────────────────────
    m = re.search(r'ARTICLE\s+(\d+)\s*(?:THEREOF|TFEU|TEEU|OF THE TREATY ON THE FUNCTIONING)', text_u)
    if m:
        f['tfeu_article'] = int(m.group(1))

    # ── Tipo di obbligo primario ──────────────────────────────────────────────
    if re.search(r'SHALL BE PROHIBITED|IS HEREBY PROHIBITED', text_u):
        f['primary_obligation'] = 'prohibition'
    elif re.search(r'THE COURT RULES|THE ACTION IS DISMISSED|ANNULS THE|DECLARES THAT', text_u):
        f['primary_obligation'] = 'judicial_ruling'
    elif re.search(r'HEREBY RECOMMENDS|SHOULD(?!\s+ENSURE)|IS ENCOURAGED', text_u[:300].upper()):
        f['primary_obligation'] = 'recommendation'
    elif re.search(r'SHALL BE CALCULATED|THE FORM SET OUT|AS SET OUT IN THE ANNEX|STANDARD FORM|TECHNICAL SPECIFICATION', text_u):
        f['primary_obligation'] = 'technical_standard'
    elif re.search(r'MEMBER STATES SHALL|IS HEREBY ESTABLISHED|SHALL ENSURE|SHALL APPLY', text_u):
        f['primary_obligation'] = 'obligation'

    # ── Allegati tecnici ──────────────────────────────────────────────────────
    f['has_technical_annex'] = bool(
        re.search(r'STANDARD FORM|THE LIST SET OUT|AS SET OUT IN ANNEX|TEMPLATE|TECHNICAL SPECIFICATIONS', text_u)
    )

    return f


print("Estrazione feature via regex...")
regex_results = nodes.apply(lambda r: extract_regex_features(r.get('title'), r.get('preamble')), axis=1)
regex_df = pd.DataFrame(list(regex_results))
for col in regex_df.columns:
    nodes[f'rx_{col}'] = regex_df[col].values

print("Copertura regex:")
for col in ['rx_author','rx_procedure','rx_tfeu_article','rx_primary_obligation']:
    n = nodes[col].notna().sum()
    print(f"  {col:<30} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)")

print("\nDistribuzione author (regex):")
print(nodes['rx_author'].value_counts(dropna=False).to_string())

Estrazione feature via regex...
Copertura regex:
  rx_author                       4121 / 4904  (84.0%)
  rx_procedure                    1208 / 4904  (24.6%)
  rx_tfeu_article                 1170 / 4904  (23.9%)
  rx_primary_obligation             69 / 4904  (1.4%)

Distribuzione author (regex):
rx_author
EP_Council                 1529
Council                     958
NaN                         783
Commission_implementing     421
Treaty                      383
Commission_other            375
Commission_delegated        288
Court                       167


## 3. Completamento Feature via LLM

Il LLM viene chiamato **solo** per i nodi in cui regex non ha trovato
`author` o `procedure`. Usa domande a risposta chiusa: non classifica,
risponde a domande precise su testo breve.

In [ ]:
from openai import OpenAI, RateLimitError
client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1" 
)

SYSTEM_PROMPT = """Sei un esperto di tecnica legislativa UE.
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza backtick."""

def llm_extract(title, preamble):
    title_s    = str(title)    if pd.notna(title)    else ''
    preamble_s = str(preamble) if pd.notna(preamble) else ''
    if not title_s and not preamble_s:
        return None, 'no_text'

    prompt = f"""Analizza questo atto normativo UE.

TESTO:
{title_s}
{preamble_s[:500]}

Rispondi SOLO con questo JSON (scegli tra i valori indicati, null se non determinabile):
{{
  "author": "EP_Council / Council / Commission_delegated / Commission_implementing / Commission_other / Court / Treaty / Other",
  "procedure": "ordinary_legislative / delegated_implementing / infringement / recommendation / judgment / treaty / other",
  "tfeu_article": numero intero o null,
  "primary_obligation": "prohibition / obligation / technical_standard / recommendation / judicial_ruling / other"
}}"""

    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp    = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role':'system','content':SYSTEM_PROMPT},
                          {'role':'user','content':prompt}],
                temperature=0, max_tokens=200,
            )
            content = resp.choices[0].message.content.strip()
            content = content.replace('```json','').replace('```','').strip()
            parsed  = json.loads(content)
            art = parsed.get('tfeu_article')
            if art is not None:
                try:    parsed['tfeu_article'] = int(str(art).strip())
                except: parsed['tfeu_article'] = None
            return parsed, 'ok'
        except json.JSONDecodeError:
            if attempt < LLM_MAX_RETRIES - 1: time.sleep(1)
        except RateLimitError:
            time.sleep(30)
        except Exception as e:
            print(f"  ERRORE: {type(e).__name__}: {e}")
            return None, 'error'
   

# Test
test_row = nodes[nodes['Label'] == '32019R0452'].iloc[0]
result, status = llm_extract(test_row.get('title'), test_row.get('preamble'))
print(f"Test LLM su 32019R0452: {status}")
print(json.dumps(result, indent=2))

Test LLM su 32019R0452: ok
{
  "author": "EP_Council",
  "procedure": "ordinary_legislative",
  "tfeu_article": 207,
  "primary_obligation": "obligation"
}


In [27]:
# Nodi da completare: mancano author O procedure
needs_llm = nodes[
    (nodes['rx_author'].isna() | nodes['rx_procedure'].isna()) &
    (nodes['title'].notna() | nodes['preamble'].notna())
].copy()

CHKPT_COLS = ['Id','llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation','llm_status']
if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi già processati")
else:
    checkpoint   = pd.DataFrame(columns=CHKPT_COLS)
    already_done = set()

nodes_todo = needs_llm[~needs_llm['Id'].isin(already_done)]
print(f"Da completare con LLM: {len(nodes_todo)}")
print(f"Tempo stimato: ~{len(nodes_todo) * LLM_DELAY / 60:.0f} minuti")

results = []
n_ok = n_err = 0

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    parsed, status = llm_extract(row.get('title'), row.get('preamble'))
    if status == 'ok': n_ok += 1
    else:              n_err += 1

    results.append({
        'Id':                     row['Id'],
        'llm_author':             parsed.get('author')             if parsed else None,
        'llm_procedure':          parsed.get('procedure')          if parsed else None,
        'llm_tfeu_article':       parsed.get('tfeu_article')       if parsed else None,
        'llm_primary_obligation': parsed.get('primary_obligation') if parsed else None,
        'llm_status':             status,
    })

    if (i+1) % 10 == 0 or (i+1) == len(nodes_todo):
        print(f"  [{i+1:>4}/{len(nodes_todo)}] {(i+1)/len(nodes_todo)*100:5.1f}%  ok:{n_ok}  err:{n_err}")

    if (i+1) % CHECKPOINT_EVERY == 0:
        batch = pd.DataFrame(results)
        chkpt = pd.concat([checkpoint, batch]).drop_duplicates('Id')
        chkpt.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint ({len(chkpt)} totali)")

    time.sleep(LLM_DELAY)

if results:
    batch = pd.DataFrame(results)
    pd.concat([checkpoint, batch]).drop_duplicates('Id').to_csv(checkpoint_file, index=False)

print(f"\nCompletato. OK: {n_ok}  Errori: {n_err}")

Da completare con LLM: 3018
Tempo stimato: ~15 minuti
  [  10/3018]   0.3%  ok:10  err:0
  [  20/3018]   0.7%  ok:20  err:0
  [  30/3018]   1.0%  ok:30  err:0
  [  40/3018]   1.3%  ok:40  err:0
  [  50/3018]   1.7%  ok:50  err:0
  --> Checkpoint (50 totali)
  [  60/3018]   2.0%  ok:60  err:0
  [  70/3018]   2.3%  ok:70  err:0
  [  80/3018]   2.7%  ok:80  err:0
  [  90/3018]   3.0%  ok:90  err:0
  [ 100/3018]   3.3%  ok:100  err:0
  --> Checkpoint (100 totali)
  [ 110/3018]   3.6%  ok:110  err:0
  [ 120/3018]   4.0%  ok:120  err:0
  [ 130/3018]   4.3%  ok:130  err:0
  [ 140/3018]   4.6%  ok:140  err:0
  [ 150/3018]   5.0%  ok:150  err:0
  --> Checkpoint (150 totali)
  [ 160/3018]   5.3%  ok:160  err:0
  [ 170/3018]   5.6%  ok:170  err:0
  [ 180/3018]   6.0%  ok:180  err:0
  [ 190/3018]   6.3%  ok:190  err:0
  [ 200/3018]   6.6%  ok:200  err:0
  --> Checkpoint (200 totali)
  [ 210/3018]   7.0%  ok:210  err:0
  [ 220/3018]   7.3%  ok:220  err:0
  [ 230/3018]   7.6%  ok:230  err:0
  [ 240/

## 3b. Valutazione Accuratezza LLM

Usa il checkpoint già prodotto — **zero chiamate API extra**.

Il regex è il riferimento: per i nodi in cui regex ha trovato `author` con certezza
ma LLM è stato chiamato comunque (perché mancava `procedure`), possiamo confrontare
`rx_author` (ground truth deterministico) con `llm_author` (risposta LLM).

Stesso ragionamento per `procedure` e `primary_obligation`.

In [ ]:
# ── Valutazione accuratezza LLM su campo author ───────────────────────────────
# Campione di validazione: nodi dove ENTRAMBI hanno un valore
# (regex ha trovato → ground truth; LLM è stato chiamato → predizione)

if not os.path.exists(checkpoint_file):
    print("[SKIP] Checkpoint non trovato. Eseguire prima la cella 3.")
else:
    llm_check = pd.read_csv(checkpoint_file)
    # Unisce con regex results (già calcolate in cella 2)
    eval_df = nodes[['Id','rx_author','rx_procedure','rx_primary_obligation']].merge(
        llm_check[['Id','llm_author','llm_procedure','llm_primary_obligation','llm_status']],
        on='Id', how='inner'
    )
    eval_df = eval_df[eval_df['llm_status'] == 'ok']

    print(f"Nodi con risposta LLM valida nel checkpoint: {len(eval_df)}")
    print()

    def accuracy_report(field_regex, field_llm, label):
        """Calcola accuratezza LLM dove regex ha ground truth."""
        both = eval_df[eval_df[field_regex].notna() & eval_df[field_llm].notna()].copy()
        if len(both) == 0:
            print(f"{label}: nessun caso con entrambi i valori — impossibile valutare")
            return
        # Normalizzazione case-insensitive
        match = (both[field_regex].str.lower().str.strip() ==
                 both[field_llm].str.lower().str.strip())
        acc = match.mean()
        n   = len(both)
        print(f"{label}")
        print(f"  Campione di confronto: {n} nodi")
        print(f"  Accordo regex ↔ LLM:  {match.sum()} / {n}  ({acc*100:.1f}%)")
        if acc >= 0.90:
            print(f"  ✓ Affidabilità alta   (≥90%)")
        elif acc >= 0.75:
            print(f"  ~ Affidabilità media  (75-90%) — usare con cautela")
        else:
            print(f"  ✗ Affidabilità bassa  (<75%) — considerare peso ridotto nella feature matrix")
        # Errori più frequenti
        errors = both[~match][[field_regex, field_llm]]
        if len(errors) > 0:
            print(f"  Discordanze più frequenti:")
            top_errors = (errors.groupby([field_regex, field_llm])
                          .size().sort_values(ascending=False).head(5))
            for (r, l), cnt in top_errors.items():
                print(f"    regex={r:<30}  llm={l:<30}  ({cnt}x)")
        print()

    accuracy_report('rx_author',             'llm_author',             'AUTHOR')
    accuracy_report('rx_procedure',           'llm_procedure',           'PROCEDURE')
    accuracy_report('rx_primary_obligation',  'llm_primary_obligation',  'PRIMARY_OBLIGATION')

    # ── Nota metodologica sul campione ───────────────────────────────────────────
    print("Note metodologiche:")
    print("  - Il campione include solo nodi dove LLM è stato chiamato (rx_author o rx_procedure mancante).")
    print("  - Non è un campione casuale del corpus: è bias verso atti con struttura testuale meno standard.")
    print("  - L'accuratezza reale sui nodi LLM-only (senza rx ground truth) potrebbe differire.")
    print("  - Per author: il regex copre pattern inequivocabili → alta concordanza attesa (>90%).")
    print("  - Per primary_obligation: il regex copre <2% dei casi → campione di confronto molto piccolo.")

## 4. Merge Feature Finali

Regex ha priorità (deterministico). LLM riempie solo i campi rimasti vuoti.

`primary_obligation` viene tenuta **separata per fonte**: `primary_obligation_regex`
e `primary_obligation_llm`. Nella feature matrix riceveranno pesi diversi (×2 vs ×0.5)
perché la copertura regex è solo 1.4% — assegnare lo stesso peso sarebbe amplificare
il rumore LLM con la stessa forza del segnale deterministico.

In [28]:
if os.path.exists(checkpoint_file):
    llm_df = pd.read_csv(checkpoint_file)
    nodes  = nodes.merge(
        llm_df[['Id','llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation']],
        on='Id', how='left'
    )
else:
    for c in ['llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation']:
        nodes[c] = None

# Regex vince, LLM riempie i buchi
nodes['author']             = nodes['rx_author'].fillna(nodes['llm_author'])
nodes['procedure']          = nodes['rx_procedure'].fillna(nodes['llm_procedure'])
nodes['tfeu_article']       = nodes['rx_tfeu_article'].fillna(nodes['llm_tfeu_article'])
nodes['has_technical_annex'] = nodes['rx_has_technical_annex'].fillna(False)

# primary_obligation: mantiene le due fonti SEPARATE
# - primary_obligation_regex: deterministico, alta affidabilità, copertura 1.4%
# - primary_obligation_llm:   LLM, affidabilità media, copertura ~60%
# Riceveranno pesi diversi nella feature matrix (×2 vs ×0.5).
# La colonna unificata 'primary_obligation' è mantenuta solo per display/export.
nodes['primary_obligation_regex'] = nodes['rx_primary_obligation']
nodes['primary_obligation_llm']   = nodes['llm_primary_obligation']
nodes['primary_obligation']       = nodes['rx_primary_obligation'].fillna(nodes['llm_primary_obligation'])

print("Copertura feature funzionali (dopo merge):")
for col in ['author','procedure','tfeu_article',
            'primary_obligation','primary_obligation_regex','primary_obligation_llm']:
    n = nodes[col].notna().sum()
    src = '← regex+llm unified' if col == 'primary_obligation' else \
          '← deterministico'     if col == 'primary_obligation_regex' else \
          '← LLM only'           if col == 'primary_obligation_llm' else ''
    print(f"  {col:<30} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)  {src}")

print("\nDistribuzione author:")
print(nodes['author'].value_counts(dropna=False).to_string())

Copertura feature funzionali (dopo merge):
  author                     4225 / 4904  (86.2%)
  procedure                  4225 / 4904  (86.2%)
  tfeu_article               1753 / 4904  (35.7%)
  primary_obligation         3025 / 4904  (61.7%)

Distribuzione author:
author
EP_Council                 1530
Council                     963
NaN                         679
Commission_implementing     422
Treaty                      387
Commission_other            378
Commission_delegated        288
Court                       210
Other                        47


In [57]:
# ── CELLA 4 — aggiunta dopo il merge regex → LLM ─────────────────────────────
# Terzo livello di fallback: inferisce author e procedure direttamente
# dal codice CELEX, deterministicamente, senza testo né API.
#
# Struttura CELEX: {settore}{anno}{tipo}{numero}
# Settore 1 = trattati, settore 3 = legislazione secondaria, settore 6 = giurisprudenza
# Tipo: R=Regulation, L=Directive, D=Decision, H=Recommendation,
#       F=Framework Decision, B=Legislative Act, E=articolo trattato
#       CJ=Court of Justice, TJ=General Court, FT=Civil Service Tribunal

CELEX_AUTHOR_MAP = {
    # Settore 1 — trattati
    '1': 'Treaty',
    # Settore 6 — giurisprudenza
    '6': 'Court',
    # Settore 3 — legislazione secondaria, per tipo
    'R':  'EP_Council',      # Regulation — quasi sempre codecisione
    'L':  'EP_Council',      # Directive
    'F':  'Council',         # Framework Decision (pre-Lisbona)
    'B':  'EP_Council',      # Legislative Act
    'D':  'Council',         # Decision — default Council, LLM già avrà provato
    'H':  'Commission_other',# Recommendation
    'O':  'Commission_other',# Guidelines
}

CELEX_PROCEDURE_MAP = {
    '1':  'treaty',
    '6':  'judgment',
    'R':  'ordinary_legislative',
    'L':  'ordinary_legislative',
    'F':  'delegated_implementing',  # framework decisions pre-Lisbona
    'B':  'ordinary_legislative',
    'D':  'other',
    'H':  'recommendation',
    'O':  'recommendation',
}


def infer_from_celex(celex):
    """
    Inferisce author e procedure dal codice CELEX senza testo né API.
    Restituisce (author, procedure) o (None, None) se il pattern non è riconoscibile.

    Pattern CELEX:
      - Settore 1 (trattati):        1{anno}E{numero}   → Treaty
      - Settore 3 (leg. secondaria): 3{anno}{tipo}{num} → vedi mappa
      - Settore 6 (giurisprudenza):  6{anno}CJ{numero}  → Court
    """
    if pd.isna(celex):
        return None, None

    c = str(celex).strip().upper()

    # Settore 1 — trattati
    if c.startswith('1'):
        return 'Treaty', 'treaty'

    # Settore 6 — giurisprudenza
    if c.startswith('6'):
        return 'Court', 'judgment'

    # Settore 3 — legislazione secondaria
    if c.startswith('3') and len(c) >= 5:
        # Pattern: 3{4_cifre_anno}{tipo_lettera/e}{numero}
        type_char = c[5] if len(c) > 5 else None  # es. 32019R0452 → 'R'

        # Caso speciale: Commission delegated/implementing
        # CELEX delegated: tipo 'R' con prefisso 'D' nel numero es. 32019R2088 no,
        # si riconosce dal titolo — non inferibile solo dal CELEX puro
        # Usiamo il tipo grezzo

        author    = CELEX_AUTHOR_MAP.get(type_char)
        procedure = CELEX_PROCEDURE_MAP.get(type_char)

        return author, procedure

    return None, None


# ── Applica CELEX inference ai nodi ancora senza author o procedure ───────────
print("Applicazione CELEX inference...")

celex_results = nodes['Label'].apply(infer_from_celex)
nodes['celex_author']    = [r[0] for r in celex_results]
nodes['celex_procedure'] = [r[1] for r in celex_results]

# Merge a tre livelli: regex → LLM → CELEX inference
nodes['author']    = (nodes['rx_author']
                      .fillna(nodes['llm_author'])
                      .fillna(nodes['celex_author']))

nodes['procedure'] = (nodes['rx_procedure']
                      .fillna(nodes['llm_procedure'])
                      .fillna(nodes['celex_procedure']))

# tfeu_article e primary_obligation: CELEX non aggiunge info qui
# rimangono regex → LLM come prima

# ── Tracciabilità: fonte di ogni valore di author ─────────────────────────────
# 'regex'   → deterministico, alta affidabilità
# 'llm'     → LLM su testo breve, affidabilità media (non validata su campione)
# 'celex'   → inferito dal codice CELEX, affidabilità variabile:
#             - Trattati (settore 1) e Giurisprudenza (settore 6): alta
#             - Legislazione secondaria: attenzione per pre-Lisbona (2009) e
#               per Regulation delegati/implementativi (indistinguibili dal CELEX)
# 'missing' → nessuna fonte disponibile, escluso dal clustering
def get_author_source(row):
    if pd.notna(row.get('rx_author')):   return 'regex'
    if pd.notna(row.get('llm_author')):  return 'llm'
    if pd.notna(row.get('celex_author')): return 'celex'
    return 'missing'

nodes['author_source'] = nodes.apply(get_author_source, axis=1)

print("Distribuzione fonte author:")
print(nodes['author_source'].value_counts().to_string())
print()
# Avviso su potenziale bias CELEX per atti pre-Lisbona
celex_nodes = nodes[nodes['author_source'] == 'celex']
pre_lisbon  = celex_nodes[celex_nodes['year'].fillna(9999).astype(int) < 2009] if 'year' in nodes.columns else pd.DataFrame()
if len(pre_lisbon) > 0:
    print(f"[ATTENZIONE] {len(pre_lisbon)} atti con author=celex e anno < 2009.")
    print("  Per questi atti l'inferenza EP_Council/Council può essere imprecisa")
    print("  (codecisione non era ancora procedura ordinaria prima del Trattato di Lisbona).")

print("\nCopertura feature funzionali (dopo CELEX inference):")
for col in ['author', 'procedure', 'tfeu_article', 'primary_obligation']:
    n = nodes[col].notna().sum()
    print(f"  {col:<25} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)")

print("\nDistribuzione author finale:")
print(nodes['author'].value_counts(dropna=False).to_string())

# Verifica: quanti nodi erano unclassifiable e ora hanno author?
if 'raw_cluster' in nodes.columns:
    prev_missing = nodes[nodes['author'].notna() &
                         (nodes['text_status'] != 'ok')]
    print(f"\nNodi senza testo ma con author inferito da CELEX: {len(prev_missing)}")

Applicazione CELEX inference...
Copertura feature funzionali (dopo CELEX inference):
  author                     4793 / 4904  (97.7%)
  procedure                  4793 / 4904  (97.7%)
  tfeu_article               1753 / 4904  (35.7%)
  primary_obligation         3025 / 4904  (61.7%)

Distribuzione author finale:
author
EP_Council                 1542
Council                    1031
Treaty                      834
Commission_implementing     422
Commission_other            405
Commission_delegated        288
Court                       224
NaN                         111
Other                        47

Nodi senza testo ma con author inferito da CELEX: 568


## 5. Embedding Semantici su Testo Breve

Solo titolo + prime 200 caratteri del preambolo. Questo testo contiene
la firma funzionale dell'atto (chi adotta, con quale procedura) senza
aggiungere contenuto tematico che disturberebbe il clustering.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

print("Caricamento modello sentence-transformers (all-mpnet-base-v2)...")
model = SentenceTransformer(EMBEDDING_MODEL)

def build_short_text(title, preamble):
    t = str(title)    if pd.notna(title)    else ''
    p = str(preamble) if pd.notna(preamble) else ''
    combined = f"{t} {p[:200]}".strip()
    return combined if len(combined) > 10 else None

nodes['short_text'] = nodes.apply(lambda r: build_short_text(r.get('title'), r.get('preamble')), axis=1)

has_text   = nodes['short_text'].notna()
texts      = nodes.loc[has_text, 'short_text'].tolist()
text_idx   = nodes.index[has_text].tolist()

print(f"Nodi con testo: {len(texts)} / {len(nodes)}")
print("Calcolo embedding...")

embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# Flag: 1.0 se il nodo ha embedding reale, 0.0 se sarà azzerato.
# Viene aggiunto come feature binaria per segnalare al UMAP quali nodi
# hanno un segnale semantico autentico (invece di usare imputazione media,
# che attrarre artificialmente i nodi senza testo verso il centroide).
nodes['has_embedding'] = 0.0
nodes.loc[has_text, 'has_embedding'] = 1.0

# Matrice completa: righe con testo → embedding reale, righe senza → zero.
# Zero-imputation è preferibile a mean-imputation: non distorce la distribuzione
# degli embedding verso il centroide artificiale degli atti con testo.
emb_matrix = np.zeros((len(nodes), embeddings.shape[1]))
for i, idx in enumerate(text_idx):
    emb_matrix[idx] = embeddings[i]

# PCA 50d — solo sui nodi con embedding reale per evitare che gli zeri
# distorcano la direzione dei componenti principali.
pca = PCA(n_components=50, random_state=42)
pca.fit(emb_matrix[nodes['has_embedding'].values == 1.0])
emb_pca = pca.transform(emb_matrix)   # proietta tutto (inclusi gli zeri)

# Azzera le righe dei nodi senza testo nello spazio PCA
# (la proiezione di uno zero-vector può produrre valori residui non nulli
#  a causa del centering interno di PCA)
emb_pca[nodes['has_embedding'].values == 0.0] = 0.0

print(f"Nodi con embedding reale: {int(nodes['has_embedding'].sum())} / {len(nodes)}")
print(f"Shape dopo PCA: {emb_pca.shape}")

Caricamento modello sentence-transformers (all-mpnet-base-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\claud\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Nodi con testo: 4225 / 4904
Calcolo embedding...


Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Shape dopo PCA: (4904, 50)


## 6. Costruzione Matrice Feature Completa

Combina le tre famiglie di feature con pesi differenziati.

Feature strutturali (×2): 9 feature tipate, normalizzate temporalmente, con HITS e k-core  
Feature funzionali (×2): `author`, `procedure` (one-hot)  
`primary_obligation_regex` (×2): deterministico, copertura 1.4%  
`primary_obligation_llm` (×0.5): LLM, copertura ~60%, segnale soft  
TFEU bucket (×1.5): bucketing dell'articolo TFUE di base giuridica  
`has_technical_annex` (×1): flag allegati tecnici  
`has_embedding` (×1): segnala se il nodo ha embedding semantico reale  
Embedding semantici PCA-50 (×1): `all-mpnet-base-v2` su titolo + 200c preambolo

In [29]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

# ── Strutturali ───────────────────────────────────────────────────────────────
# Feature selezionate per minimizzare ridondanza e massimizzare segnale gerarchico.
# Le versioni _norm rimuovono il bias temporale (atti vecchi hanno più citazioni).
struct_feature_cols = [
    'indegree_norm',                  # citazioni ricevute vs contemporanei
    'pagerank_norm',                  # autorità globale vs contemporanei
    'hits_authority_norm',            # HITS authority vs contemporanei
    'hierarchical_authority_norm',    # BASED_ON ricevuti vs contemporanei
    'hits_ratio',                     # authority/hub — apicale vs terminale
    'hierarchical_ratio',             # BASED_ON_received / BASED_ON_sent
    'citation_ratio',                 # indegree / outdegree generico
    'kcore',                          # posizione nel nucleo topologico
    'lateral_activity',               # attività di modifica
]
struct_feature_cols = [c for c in struct_feature_cols if c in nodes.columns]
struct_data = nodes[struct_feature_cols].values.astype(float)
# Log1p su feature con distribuzione power-law
for i, col in enumerate(struct_feature_cols):
    if col in ('indegree_norm','pagerank_norm','hits_authority_norm',
               'hierarchical_authority_norm','kcore','lateral_activity'):
        struct_data[:, i] = np.log1p(struct_data[:, i])
struct_scaled = StandardScaler().fit_transform(struct_data)
print(f"Feature strutturali: {struct_scaled.shape[1]} colonne")

# ── Funzionali (categorici → one-hot) ────────────────────────────────────────
# author e procedure: deterministici (regex) → peso ×2
func_cat_strong = nodes[['author','procedure']].fillna('unknown')
func_encoded_strong = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(func_cat_strong)

# primary_obligation_regex: copertura 1.4% ma completamente deterministico → peso ×2
# Tenuto separato da LLM perché mescolarli nasconde la differenza di affidabilità.
pobl_regex_cat = nodes[['primary_obligation_regex']].fillna('unknown')
pobl_regex_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(pobl_regex_cat)

# primary_obligation_llm: copertura ~60% ma fonte LLM non validata sistematicamente → peso ×0.5
# Contribuisce come segnale soft: orienta il clustering senza dominarlo.
pobl_llm_cat = nodes[['primary_obligation_llm']].fillna('unknown')
pobl_llm_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(pobl_llm_cat)

# TFEU article: bucketing per tipo di competenza
def tfeu_bucket(art):
    if pd.isna(art): return 'unknown'
    a = int(art)
    if   a <= 17:  return 'principles'
    elif a <= 66:  return 'internal_market'
    elif a <= 113: return 'policies'
    elif a <= 197: return 'institutions'
    elif a <= 291: return 'implementing'
    else:          return 'other'

nodes['tfeu_bucket'] = nodes['tfeu_article'].apply(tfeu_bucket)
tfeu_encoded = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(nodes[['tfeu_bucket']])

annex_feat    = nodes['has_technical_annex'].astype(float).values.reshape(-1,1)
has_emb_feat  = nodes['has_embedding'].values.reshape(-1,1)

# ── Concatenazione con pesi ───────────────────────────────────────────────────
feature_matrix = np.hstack([
    struct_scaled       * 2.0,   # strutturali: segnale gerarchico forte
    func_encoded_strong * 2.0,   # author + procedure: deterministici
    pobl_regex_enc      * 2.0,   # primary_obligation regex: deterministico
    pobl_llm_enc        * 0.5,   # primary_obligation LLM: segnale soft
    tfeu_encoded        * 1.5,
    annex_feat          * 1.0,
    has_emb_feat        * 1.0,
    emb_pca             * 1.0,
])

print(f"Matrice feature: {feature_matrix.shape}")
print(f"  strutturali (×2):           {struct_scaled.shape[1]} col  {struct_feature_cols}")
print(f"  author+procedure (×2):      {func_encoded_strong.shape[1]} col")
print(f"  primary_oblig regex (×2):   {pobl_regex_enc.shape[1]} col  [{nodes['primary_obligation_regex'].notna().sum()} nodi con valore reale]")
print(f"  primary_oblig llm (×0.5):   {pobl_llm_enc.shape[1]} col  [{nodes['primary_obligation_llm'].notna().sum()} nodi con valore reale]")
print(f"  TFEU bucket (×1.5):         {tfeu_encoded.shape[1]} col")
print(f"  annex (×1):                 1 col")
print(f"  has_embedding (×1):         1 col")
print(f"  embedding PCA-50 (×1):      {emb_pca.shape[1]} col")

Matrice feature: (4904, 86)
  strutturali:  4 × 2
  funzionali:   24 × 2
  TFEU bucket:  7 × 1.5
  annex:        1
  embedding:    50


## 7. UMAP — Riduzione Dimensionale

In [30]:
import umap

print(f"UMAP: {feature_matrix.shape[1]}d → {UMAP_N_COMPONENTS}d  (può richiedere 2-5 minuti)")

reducer = umap.UMAP(
    n_components=UMAP_N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST, metric='euclidean', random_state=UMAP_RANDOM_STATE,
)
emb_umap = reducer.fit_transform(feature_matrix)
print(f"Shape dopo UMAP: {emb_umap.shape}")

# Embedding 2D separato per visualizzazione
reducer_2d = umap.UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
                        min_dist=0.1, random_state=UMAP_RANDOM_STATE)
emb_2d = reducer_2d.fit_transform(feature_matrix)
nodes['umap_x'] = emb_2d[:, 0]
nodes['umap_y'] = emb_2d[:, 1]
print("Embedding 2D per visualizzazione: ok")

UMAP: 86d → 15d  (può richiedere 2-5 minuti)


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape dopo UMAP: (4904, 15)


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedding 2D per visualizzazione: ok


## 8. HDBSCAN — Clustering Non Supervisionato

Il numero di cluster (= layer) emerge dai dati.
Gli atti con label `-1` (noise) non appartengono a nessun cluster:
sono strutturalmente anomali e meritano analisi separata.

In [58]:
# ── Separazione nodi con/senza dati funzionali ───────────────────────────────
# I nodi senza author (text_status != 'ok', regex e LLM falliti) non vengono
# clusterizzati: condividono lo stesso profilo non perché hanno lo stesso ruolo
# normativo, ma perché mancano di dati. Includerli creerebbe un cluster
# artefatto. Vengono assegnati a categoria separata (-2) e trattati come
# "unclassifiable" nel report finale — distinti dal noise semantico (-1).
import hdbscan

has_functional = nodes['author'].notna()
nodes_valid    = nodes[has_functional].copy()
nodes_missing  = nodes[~has_functional].copy()

emb_umap_valid = emb_umap[has_functional.values]

print(f"Nodi con dati funzionali:    {len(nodes_valid)} ({len(nodes_valid)/len(nodes)*100:.1f}%)")
print(f"Nodi senza dati funzionali:  {len(nodes_missing)} ({len(nodes_missing)/len(nodes)*100:.1f}%)")
print(f"  → esclusi dal clustering, etichettati come 'unclassifiable'")
print()

# ── Parametri ─────────────────────────────────────────────────────────────────

print(f"HDBSCAN — min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    prediction_data=True,
    cluster_selection_method='eom',
)
cluster_labels = clusterer.fit_predict(emb_umap_valid)

soft_valid     = hdbscan.all_points_membership_vectors(clusterer)
n_real_clusters = soft_valid.shape[1]
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print(f"\nLayer trovati:  {n_clusters}")
print(f"Atti noise:     {n_noise} ({n_noise/len(nodes_valid)*100:.1f}% dei nodi clusterizzati)")
print("\nDistribuzione:")
for lbl, cnt in zip(*np.unique(cluster_labels, return_counts=True)):
    name = 'noise' if lbl == -1 else f'cluster_{lbl}'
    print(f"  {name:<15} {cnt:>5}  ({cnt/len(nodes_valid)*100:.1f}%)")

# ── Ricomposizione dataframe completo ─────────────────────────────────────────
nodes_valid['raw_cluster']   = cluster_labels
nodes_missing['raw_cluster'] = -2   # unclassifiable — dati insufficienti

# Invece di sort_index(), preserva l'ordine esplicito
nodes_valid['_original_order']   = np.where(has_functional.values)[0]
nodes_missing['_original_order'] = np.where(~has_functional.values)[0]

nodes = pd.concat([nodes_valid, nodes_missing])
nodes = nodes.sort_values('_original_order').drop(columns='_original_order')
nodes = nodes.reset_index(drop=True)

# Soft matrix: ricostruisci sull'indice originale
soft = np.zeros((len(nodes), n_real_clusters))
for i, pos in enumerate(np.where(has_functional.values)[0]):
    soft[pos] = soft_valid[i]

print(f"\nSoft membership matrix: {soft.shape}")
print(f"  (righe con tutti zero = nodi unclassifiable: {len(nodes_missing)})")

Nodi con dati funzionali:    4793 (97.7%)
Nodi senza dati funzionali:  111 (2.3%)
  → esclusi dal clustering, etichettati come 'unclassifiable'

HDBSCAN — min_cluster_size=200, min_samples=30

Layer trovati:  10
Atti noise:     996 (20.8% dei nodi clusterizzati)

Distribuzione:
  noise             996  (20.8%)
  cluster_0         335  (7.0%)
  cluster_1         271  (5.7%)
  cluster_2         346  (7.2%)
  cluster_3         526  (11.0%)
  cluster_4         238  (5.0%)
  cluster_5         228  (4.8%)
  cluster_6         237  (4.9%)
  cluster_7         682  (14.2%)
  cluster_8         354  (7.4%)
  cluster_9         580  (12.1%)

Soft membership matrix: (4904, 10)
  (righe con tutti zero = nodi unclassifiable: 111)


## 9. Ordinamento Gerarchico dei Cluster

Ordina i cluster da apicale a terminale usando due segnali:
1. `citation_ratio` medio per cluster (apicale = citato da molti, cita pochi)
2. Citazioni inter-cluster ricevute (apicale = tutti lo citano)

In [59]:
valid = nodes[nodes['raw_cluster'] >= 0].copy()

stats = valid.groupby('raw_cluster').agg(
    n_atti        =('Id','count'),
    mean_indegree =('indegree','mean'),
    mean_outdegree=('outdegree','mean'),
    mean_ratio    =('citation_ratio','mean'),
    mean_pagerank =('pagerank','mean'),
).round(3)

# Flusso citazioni inter-cluster
id_to_cluster = dict(zip(nodes['Id'], nodes['raw_cluster']))
received = {}
for _, edge in edges.iterrows():
    c_src = id_to_cluster.get(edge[src_col], -1)
    c_tgt = id_to_cluster.get(edge[tgt_col], -1)
    if c_src != -1 and c_tgt != -1 and c_src != c_tgt:
        received[c_tgt] = received.get(c_tgt, 0) + 1

stats['received_citations'] = [received.get(c, 0) for c in stats.index]

# Score gerarchico combinato
scaler_mm = MinMaxScaler()
stats['score_ratio']    = scaler_mm.fit_transform(stats[['mean_ratio']])
stats['score_received'] = scaler_mm.fit_transform(stats[['received_citations']])
stats['hierarchy_score'] = stats['score_ratio'] * HIERARCHY_W_RATIO + stats['score_received'] * HIERARCHY_W_RECEIVED

stats = stats.sort_values('hierarchy_score', ascending=False)
stats['layer_rank']  = range(1, len(stats)+1)
stats['layer_label'] = stats['layer_rank'].apply(lambda r: f'L{r}')

print("Cluster ordinati (L1 = più apicale):")
print(stats[['layer_label','n_atti','mean_indegree','mean_outdegree','mean_ratio','received_citations','hierarchy_score']].to_string())

Cluster ordinati (L1 = più apicale):
            layer_label  n_atti  mean_indegree  mean_outdegree  mean_ratio  received_citations  hierarchy_score
raw_cluster                                                                                                    
7                    L1     682         21.674          15.732      10.258                6311         1.000000
0                    L2     335          5.000           0.913       4.832                1486         0.357210
9                    L3     580          3.276           4.948       0.970                1203         0.104758
6                    L4     237          2.565           2.743       1.756                 416         0.101178
5                    L5     228          1.132           1.167       0.909                 258         0.039568
8                    L6     354          1.915           4.133       0.575                 368         0.026486
2                    L7     346          1.737           2.962     

## 9b. Sensitivity Analysis — Stabilità dell'Ordinamento

I pesi del `hierarchy_score` sono configurabili in cella 0 ma comunque arbitrari.
Questa cella verifica se l'identificazione di L1 e L2 cambia al variare dei pesi,
e quante inversioni di ordine si producono tra layer adiacenti.
Se l'ordinamento è stabile, i risultati non dipendono criticamente dalla scelta dei pesi.

In [ ]:
# ── Sensitivity analysis sui pesi del hierarchy_score ─────────────────────────
# Testa 5 combinazioni di pesi e mostra se L1 e l'ordinamento cambiano.
weight_configs = [
    (1.0, 0.0, 'solo ratio'),
    (0.8, 0.2, 'ratio pesante'),
    (0.6, 0.4, 'baseline (configurazione corrente)'),
    (0.4, 0.6, 'received pesante'),
    (0.0, 1.0, 'solo received'),
]

print(f"{'Pesi (r/c)':<12}  {'L1':>10}  {'L2':>10}  {'L3':>10}  Inversioni  Note")
print('-' * 75)

baseline_order = None

for w_ratio, w_recv, label in weight_configs:
    s = stats.copy()
    s['score_r'] = MinMaxScaler().fit_transform(s[['mean_ratio']])
    s['score_c'] = MinMaxScaler().fit_transform(s[['received_citations']])
    s['hs']      = s['score_r'] * w_ratio + s['score_c'] * w_recv
    s = s.sort_values('hs', ascending=False).reset_index()
    order = list(s['raw_cluster'])

    if baseline_order is None:
        baseline_order = order
        inversions = 0
    else:
        # Conta inversioni rispetto alla baseline
        pos = {c: i for i, c in enumerate(baseline_order)}
        inv = sum(1 for i in range(len(order)-1)
                  if pos[order[i]] > pos[order[i+1]])
        inversions = inv

    l1 = s.iloc[0]['layer_label'] if 'layer_label' in s.columns else f"c{order[0]}"
    l2 = s.iloc[1]['layer_label'] if 'layer_label' in s.columns else f"c{order[1]}"
    l3 = s.iloc[2]['layer_label'] if 'layer_label' in s.columns else f"c{order[2]}"
    marker = ' ← configurazione attiva' if abs(w_ratio - HIERARCHY_W_RATIO) < 1e-9 else ''
    print(f"{w_ratio:.1f}/{w_recv:.1f}      {l1:>10}  {l2:>10}  {l3:>10}  {inversions:>10}  {label}{marker}")

print()
print("Interpretazione: 0 inversioni = ordinamento stabile; >2 inversioni = ordinamento sensibile ai pesi.")
print("Le inversioni tra layer non-adiacenti (es. L1↔L3) sono più preoccupanti di quelle tra layer adiacenti.")

## 10. Calcolo Purezza e Identificazione Atti Patologici

Produce **due misure di patologia complementari**:

| Metrica | Dipende da | Cosa misura |
|---|---|---|
| `is_pathological` | HDBSCAN soft membership | Atto distribuito su più layer nello spazio delle feature |
| `is_pathological_structural` | Feature tipate (BASED_ON) | Atto con tensione apicale+subordinata nelle citazioni |
| `pathology_confidence` | Entrambe | 0=sano, 1=borderline, 2=confermato |

La metrica strutturale è indipendente da `min_cluster_size` e costituisce
il cross-check principale per la robustezza del risultato.

In [43]:
cluster_to_layer = dict(zip(stats.index, stats['layer_label']))
cluster_to_rank  = dict(zip(stats.index, stats['layer_rank']))
real_clusters    = sorted([c for c in set(cluster_labels) if c != -1])

nodes['layer']       = nodes['raw_cluster'].map(cluster_to_layer).fillna('noise')
nodes['layer_rank']  = nodes['raw_cluster'].map(cluster_to_rank).fillna(0).astype(int)

# Colonne membership per layer
membership_cols = []
for i, c in enumerate(real_clusters):
    col = f"membership_{cluster_to_layer[c]}"
    nodes[col] = soft[:, i]
    membership_cols.append(col)

# Purity
nodes['purity'] = soft.max(axis=1)

# Entropia di Shannon
def shannon_entropy(row):
    p = row[row > 0]
    return -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

nodes['layer_entropy'] = [shannon_entropy(soft[i]) for i in range(len(nodes))]

# Level span: distanza tra layer più alto e più basso con membership > soglia
def level_span(i):
    ranks = [cluster_to_rank[c] for j, c in enumerate(real_clusters)
             if soft[i, j] > MEMBERSHIP_THRESHOLD]
    return max(ranks) - min(ranks) if len(ranks) > 1 else 0

nodes['level_span'] = [level_span(i) for i in range(len(nodes))]

# Flag patologico — basato su soft membership HDBSCAN
# ATTENZIONE: questa metrica è sensibile a min_cluster_size.
# Usare in combinazione con is_pathological_structural (sotto).
nodes['is_pathological'] = (
    (nodes['purity'] < PURITY_THRESHOLD) | (nodes['level_span'] >= SPAN_THRESHOLD)
)

print("=" * 50)
print("RISULTATI")
print("=" * 50)
print(f"Layer identificati: {n_clusters}")
print(f"Atti noise:         {n_noise} ({n_noise/len(nodes)*100:.1f}%)")
print()
print("Distribuzione per layer:")
print(nodes['layer'].value_counts().sort_index().to_string())
print()
print(f"Purezza media:   {nodes['purity'].mean():.3f}")
print(f"Entropia media:  {nodes['layer_entropy'].mean():.3f}")
print()
print("Purezza media per layer:")
print(nodes.groupby('layer')['purity'].agg(['mean','median','min']).round(3).sort_index().to_string())
print()

RISULTATI
Layer identificati: 4
Atti noise:         241 (4.9%)

Distribuzione per layer:
layer
L1       2321
L2        478
L3       1265
L4        488
noise     352

Purezza media:   0.747
Entropia media:  0.421

Purezza media per layer:
        mean  median    min
layer                      
L1     0.752   0.981  0.035
L2     0.922   1.000  0.328
L3     0.783   0.835  0.201
L4     0.983   1.000  0.719
noise  0.020   0.016  0.000



## 11. Validazione Post-hoc: Composizione Istituzionale

Verifica che l'ordinamento emerso abbia
senso normativo. Se L1 contiene prevalentemente EP_Council/Treaty e l'ultimo
layer contiene prevalentemente Court, l'ordinamento è corretto.

In [61]:
layer_order = [f'L{i}' for i in range(1, n_clusters+1)] + ['noise']

for layer in layer_order:
    sub = nodes[nodes['layer'] == layer]
    if len(sub) == 0: continue
    print(f"{'─'*45}")
    print(f"{layer}  ({len(sub)} atti) — indegree: {sub['indegree'].mean():.1f}  outdegree: {sub['outdegree'].mean():.1f}")
    for author, pct in sub['author'].value_counts(normalize=True).head(4).items():
        bar = '█' * int(pct * 25)
        print(f"  {str(author):<32} {bar} {pct:.0%}")
    print(f"  purezza media: {sub['purity'].mean():.3f}  |  atti patologici: {sub['is_pathological'].sum()} ({sub['is_pathological'].mean()*100:.0f}%)")

─────────────────────────────────────────────
L1  (682 atti) — indegree: 21.7  outdegree: 15.7
  EP_Council                       ██████████████████ 74%
  Treaty                           ████ 20%
  Council                           3%
  Court                             2%
  purezza media: 0.624  |  atti patologici: 374 (55%)
─────────────────────────────────────────────
L2  (335 atti) — indegree: 5.0  outdegree: 0.9
  Treaty                           ███████████████████ 79%
  Council                          ███ 13%
  Commission_other                 █ 6%
  Court                             1%
  purezza media: 0.761  |  atti patologici: 96 (29%)
─────────────────────────────────────────────
L3  (580 atti) — indegree: 3.3  outdegree: 4.9
  Council                          █████████████████████ 86%
  Other                            █ 8%
  EP_Council                       █ 6%
  Commission_other                  0%
  purezza media: 0.580  |  atti patologici: 319 (55%)
─────────────────

## 11b. Validazione Esterna — Cross-tabulation Layer × LegalType

`LegalType` è una variabile che il modello **non ha usato** nel clustering.
Se i layer inferiti hanno senso normativo, ci aspettiamo:
- Layer apicali (L1, L2): dominanza di `Regulation` e `Directive`
- Layer mediani: `Decision`, `Recommendation`
- Layer terminali: `Case_Law`, `Legislative_Act` (sentenze, atti tecnici)

Questa è l'unica verifica con una variabile esogena al clustering.

In [ ]:
# ── Cross-tabulation layer × LegalType (validazione esterna) ─────────────────
if 'LegalType' in nodes.columns:
    classified = nodes[nodes['layer'] != 'unclassifiable'].copy()
    classified['layer'] = pd.Categorical(
        classified['layer'],
        categories=[f'L{i}' for i in range(1, n_clusters+1)] + ['noise'],
        ordered=True
    )
    crosstab = pd.crosstab(
        classified['layer'],
        classified['LegalType'],
        normalize='index'
    ).round(3)
    print("Distribuzione LegalType per layer (% di riga):")
    print(crosstab.to_string())
    print()
    # Test: Case_Law dovrebbe concentrarsi nei layer terminali (alto rank numerico)
    if 'Case_Law' in crosstab.columns:
        caselaw_peak = crosstab['Case_Law'].idxmax()
        print(f"  Case_Law più concentrata in: {caselaw_peak}")
        if caselaw_peak in [f'L{i}' for i in range(7, n_clusters+1)]:
            print("  ✓ Coerente: giurisprudenza nei layer terminali")
        else:
            print("  ✗ Attenzione: giurisprudenza non nei layer terminali — verificare ordinamento")
    if 'Regulation' in crosstab.columns:
        reg_peak = crosstab['Regulation'].idxmax()
        print(f"  Regulation più concentrata in: {reg_peak}")
        if reg_peak in ['L1','L2','L3']:
            print("  ✓ Coerente: regolamenti nei layer apicali")
        else:
            print("  ✗ Attenzione: regolamenti non nei layer apicali — verificare ordinamento")
else:
    print("[SKIP] Colonna LegalType non trovata nel dataset. Aggiungila da nodes_light.csv per la validazione esterna.")

## 11c. Analisi Bucket Noise

Il bucket `noise` contiene il 22% degli atti — un volume significativo e strutturalmente
eterogeneo (EP_Council, Court, Treaty). Questa cella distingue due cause di rumore:
1. Rumore **strutturale**: l'atto è genuinamente anomalo (nodo ponte tra layer diversi)
2. Rumore **algoritmico**: HDBSCAN con `min_cluster_size=200` è troppo conservativo
   e scarta atti che potrebbero formare cluster più piccoli ma interpretabili.

In [44]:
# ── Analisi noise: composizione e causa ──────────────────────────────────────
noise_nodes = nodes[nodes['layer'] == 'noise'].copy()
print(f"Noise: {len(noise_nodes)} atti ({len(noise_nodes)/len(nodes)*100:.1f}% del totale)")
print()

if 'LegalType' in noise_nodes.columns:
    print("LegalType nel noise:")
    print(noise_nodes['LegalType'].value_counts().to_string())
    print()

if 'author' in noise_nodes.columns:
    print("Author nel noise:")
    print(noise_nodes['author'].value_counts(dropna=False).to_string())
    print()

# Feature strutturali del noise vs atti clusterizzati
classified = nodes[nodes['layer'].isin([f'L{i}' for i in range(1, n_clusters+1)])]
print("Feature strutturali — noise vs clusterizzati:")
for feat in ['indegree','outdegree','citation_ratio','pagerank']:
    if feat in nodes.columns:
        m_noise = noise_nodes[feat].mean()
        m_class = classified[feat].mean()
        print(f"  {feat:<20}  noise: {m_noise:.3f}   clusterizzati: {m_class:.3f}")

# Test con min_cluster_size ridotto per vedere quanti noise vengono recuperati
print()
print("Test sensibilità min_cluster_size:")
print("(riesegue HDBSCAN sullo stesso embedding UMAP con parametri diversi)")
print("NOTA: questo non aggiorna i layer nel dataframe — è solo diagnostico.")
print()

for mcs in [400, 300, 200]:
    test_clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=max(20, mcs//8),
        prediction_data=False,
        cluster_selection_method='eom',
    )
    test_labels = test_clusterer.fit_predict(emb_umap_valid)
    n_c  = len(set(test_labels)) - (1 if -1 in test_labels else 0)
    n_no = (test_labels == -1).sum()
    print(f"  min_cluster_size={mcs:>4}  →  {n_c} cluster, {n_no} noise ({n_no/len(emb_umap_valid)*100:.1f}%)")

print()
print(f"Configurazione attiva (cella 0): min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}")
print("Per usare un valore diverso: modifica HDBSCAN_MIN_CLUSTER_SIZE in cella 0 e riesegui dal passo 8.")

Noise: 352 atti (7.2% del totale)

LegalType nel noise:
LegalType
Decision             159
Legislative_Act      103
Regulation            45
Directive             28
Treaty                 8
Complementary_Act      8
Case_Law               1

Author nel noise:
author
Commission_other    154
NaN                 111
EP_Council           79
Treaty                8

Feature strutturali — noise vs clusterizzati:
  indegree              noise: 2.000   clusterizzati: 5.357
  outdegree             noise: 5.517   clusterizzati: 5.085
  citation_ratio        noise: 1.030   clusterizzati: 2.696
  pagerank              noise: 0.000   clusterizzati: 0.000

Test sensibilità min_cluster_size:
(riesegue HDBSCAN sullo stesso embedding UMAP con parametri diversi)
NOTA: questo non aggiorna i layer nel dataframe — è solo diagnostico.

  min_cluster_size= 400  →  4 cluster, 241 noise (5.0%)
  min_cluster_size= 300  →  4 cluster, 92 noise (1.9%)
  min_cluster_size= 200  →  6 cluster, 339 noise (7.1%)

Config

In [56]:
unclassifiable = nodes[nodes['raw_cluster'] == -2]
print(unclassifiable['LegalType'].value_counts())
print(unclassifiable['text_status'].value_counts())

LegalType
Treaty               447
Legislative_Act      104
Decision              68
Recommendation        27
Case_Law              14
Regulation             8
Complementary_Act      8
Directive              3
Name: count, dtype: int64
text_status
no_structure    352
not_found       327
Name: count, dtype: int64


## 12. Export

In [ ]:
drop_cols = [c for c in nodes.columns 
             if c.startswith('rx_') or c.startswith('llm_')
             or c in ('short_text', 'raw_cluster', 'tfeu_bucket')]
output_df = nodes.drop(columns=drop_cols)
output_df.to_csv(output_file, index=False)

print(f"Salvato: {output_file}")
print(f"  {len(output_df)} righe, {len(output_df.columns)} colonne")
print()
new_cols = ['layer','layer_rank','purity','layer_entropy','level_span',
            'is_pathological','is_pathological_structural','pathology_confidence',
            'structural_tension','umap_x','umap_y','author','author_source',
            'procedure','tfeu_article','primary_obligation','has_technical_annex',
            'has_embedding','hierarchical_authority','hierarchical_subordination',
            'hierarchical_ratio','hits_authority','hits_hub'] + membership_cols
print("Colonne aggiunte:")
for c in new_cols:
    if c in output_df.columns: print(f"  {c}")